# Lección 02 - Explorando el Microsoft Agent Framework

El **Microsoft Agent Framework (MAF)** es un marco unificado para construir agentes de IA. Proporciona una arquitectura limpia y componible con cuatro bloques de construcción principales:

- **Cliente** – se conecta a un endpoint de modelo de IA y maneja la comunicación
- **Agente** – envuelve a un cliente con instrucciones y definiciones de herramientas
- **Herramientas** – amplían las capacidades del agente con funciones personalizadas que el modelo puede llamar
- **Sesión** – mantiene el historial de la conversación para interacciones de varios turnos

En esta lección, construiremos un **agente de reservas de viaje** que verifica la disponibilidad del destino usando estos conceptos.


## Configuración


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Entendiendo la Arquitectura del Marco de Agentes

El Marco de Agentes de Microsoft sigue una arquitectura en capas:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Cliente** – Un `AzureAIProjectAgentProvider` se conecta a una implementación de Azure OpenAI. Gestiona la autenticación, el formato de solicitudes y el análisis de respuestas.
2. **Agente** – Creado a partir del cliente mediante `provider.create_agent()`, el agente combina acceso al modelo con instrucciones (prompt del sistema) y herramientas.
3. **Herramientas** – Funciones de Python decoradas con `@tool` que el agente puede invocar para realizar acciones o recuperar datos.
4. **Sesión** – Un objeto `AgentSession` (creado mediante `agent.create_session()`) que almacena el historial de la conversación, permitiendo un diálogo de múltiples turnos donde el agente recuerda el contexto previo.

Construyamos cada capa paso a paso.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Añadiendo herramientas con el decorador @tool

Las herramientas permiten a los agentes realizar acciones más allá de generar texto. El decorador `@tool` convierte una función de Python común en algo que el agente puede llamar.

Puntos clave:
- Usa `Annotated[type, "descripción"]` para que el modelo entienda cada parámetro.
- La cadena de documentación se convierte en la descripción de la herramienta que ve el modelo.
- `approval_mode="never_require"` significa que la herramienta se ejecuta automáticamente sin confirmación del usuario.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Creando un Agente con Herramientas

Ahora combinamos el cliente, las instrucciones y las herramientas en un agente. Las `instructions` actúan como el prompt del sistema: definen la personalidad y el comportamiento del agente.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Conversaciones de varios turnos con sesiones

Un `AgentSession` (creado mediante `agent.create_session()`) realiza un seguimiento de todos los mensajes en una conversación. Al pasar la misma sesión a cada llamada `agent.run()`, el agente tiene acceso a todo el historial de la conversación y puede referirse a mensajes anteriores.

Pasamos `tools=[check_destination_availability]` para que el agente pueda llamar a nuestro verificador de disponibilidad durante cada turno.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Resumen

En esta lección exploraste los cuatro pilares del Microsoft Agent Framework:

| Concepto | Lo Que Aprendiste |
|---------|------------------|
| **Cliente** | `AzureAIProjectAgentProvider` se conecta a Azure OpenAI con autenticación basada en credenciales |
| **Agente** | `provider.create_agent()` agrupa una conexión de modelo con instrucciones y un nombre |
| **Herramientas** | El decorador `@tool` expone funciones Python para que el agente las llame |
| **Sesión** | `agent.create_session()` mantiene el historial de conversación a través de múltiples turnos |

Estos bloques fundamentales se combinan para crear agentes que pueden mantener conversaciones naturales, llamar a funciones externas y mantener el contexto — la base para patrones agenticos más avanzados en lecciones posteriores.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Descargo de responsabilidad**:  
Este documento ha sido traducido utilizando el servicio de traducción automática [Co-op Translator](https://github.com/Azure/co-op-translator). Aunque nos esforzamos por lograr precisión, tenga en cuenta que las traducciones automáticas pueden contener errores o inexactitudes. El documento original en su idioma nativo debe considerarse la fuente autorizada. Para información crítica, se recomienda una traducción profesional realizada por humanos. No nos hacemos responsables de posibles malentendidos o interpretaciones erróneas derivadas del uso de esta traducción.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
